# Mamba-CL on Split-CIFAR100 — De-focus Mamba **base** backbone

Same protocol as our reproduced large-backbone run (10 tasks, 10 epochs/task, null-space, seed 2024; AA 90.16%, FM 2.18%),
but with `defocus_mamba_base` (24 layers, 768-dim, ~92M params) instead of `defocus_mamba_large` (48 layers, 1024-dim, ~330M).
Base is the smallest pretrained De-focus Mamba released upstream.

Settings: **Accelerator = GPU T4** (fp16 AMP; P100 has no tensor cores), **Internet = on**. Kaggle sessions stop at 12 h,
so check the projected total printed after task 1; lower `EPOCHS` if it will not fit.

In [ ]:
# ---- run parameters ----
REPO     = "https://github.com/HasinthakaPiyumal/hippo-cortex.git"
BRANCH   = "feat/mamba-cl-base-backbone"
MODEL    = "defocus_mamba_base"
TASKS    = 10
EPOCHS   = 10        # Mamba-CL default; the large-backbone baseline used 10
BATCH    = 200
SEED     = 2024
USE_AMP  = True
WORKERS  = 4

In [ ]:
!nvidia-smi
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("hippo-cortex"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
ROOT = "/kaggle/working/hippo-cortex"
!cd {ROOT} && git log --oneline -1

Mamba-CL asserts the CUDA fast path, so `mamba-ssm` and `causal-conv1d` must import. Their setup fetches a prebuilt wheel matching Kaggle's torch/CUDA when one exists, otherwise compiles (can take ~20-30 min).

In [ ]:
!pip install -q timm==1.0.9 einops yacs easydict
!pip install -q --no-build-isolation causal-conv1d mamba-ssm
import torch, transformers
from transformers.utils.import_utils import is_mamba_ssm_available, is_causal_conv1d_available
print("torch", torch.__version__, "cuda", torch.version.cuda, "transformers", transformers.__version__)
assert is_mamba_ssm_available() and is_causal_conv1d_available(), "mamba fast path unavailable"
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
from causal_conv1d import causal_conv1d_fn
print("mamba fast path OK")

In [ ]:
CKPT = "/kaggle/working/defocus_mamba_base_cls.pth"
if not os.path.exists(CKPT):
    !wget -q -O {CKPT} https://github.com/OpenGVLab/De-focus-Attention-Networks/releases/download/v1.0/defocus_mamba_base_cls.pth
DATA = f"{ROOT}/data/cifar100-images"
if not os.path.exists(f"{DATA}/val"):
    !cd {ROOT} && python scripts/download_datasets.py --root data --dataset cifar100 && python scripts/split_cifar100.py
!ls -la {CKPT}; ls {DATA}/train | wc -l; ls {DATA}/val | wc -l

In [ ]:
import sys, time, re
LOG = f"/kaggle/working/mamba_cl_{MODEL}_t{TASKS}_e{EPOCHS}_s{SEED}.log"
cmd = [sys.executable, "train_eval.py", "-d", "cifar100", "-m", MODEL,
       "-t", str(TASKS), "-e", str(EPOCHS), "-b", str(BATCH),
       "--pretrained_path", CKPT, "--data_root", DATA,
       "--null_eta", "0.95", "--use_null_space", "--seed", str(SEED),
       "--use_amp", str(USE_AMP), "-jt", str(WORKERS)]
print(" ".join(cmd))
t0 = time.time(); done = 0
with open(LOG, "w") as f:
    p = subprocess.Popen(cmd, cwd=f"{ROOT}/baselines/mamba-cl", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        f.write(line); f.flush()
        if "Results of task" in line or "Error" in line or "Traceback" in line or "missing" in line.lower():
            print(line, end="")
        if "Results of task" in line:
            done += 1
            el = time.time() - t0
            print(f"   elapsed {el/3600:.2f} h, projected total {el/done*TASKS/3600:.2f} h")
    p.wait()
print("exit code", p.returncode, f"in {(time.time()-t0)/3600:.2f} h; full log: {LOG}")

In [ ]:
!tail -n 40 {LOG}